# Module 4 lab: Multi-Layer Agent Evaluation

This notebook evaluates the behavior of a customer support agent using multiple evaluation layers. Rather than focusing only on the final response, the notebook examines the complete execution process, including tool selection, tool arguments, database outcomes, and policy compliance.

The evaluation pipeline includes several stages, organized into six primary categories:

- Test Set and Ground Truth Spec: Define the expected behavior for each query.
- Trace Processing and Trajectory Extraction: Extract tool-call trajectories from execution traces.
- Tool Selection and Invocation Evaluation: Evaluate tool selection accuracy and argument precision.
- Outcome Verification: Verify database outcomes against expected execution results.
- Policy Compliance Evaluation: Enforce business rules like checking identity before issuing refunds.
- Multi-Layered Analysis and Logging: Aggregate evaluation metrics across all stages into logs.

The goal is to determine whether the agent performed the correct actions, executed them correctly, and followed required business rules.

## Table of Contents
- [Imports](#Imports)
- [1 - Test Set and Ground Truth Spec](#1---test-set--ground-truth-spec)
- [2 - Trace Processing and Trajectory Extraction](#2---trace-processing--trajectory-extraction)
- [3 - Tool Selection and Invocation Evaluation](#3---tool-selection--invocation-evaluation)
    - **[Exercise 1 - tool_selection](#exercise-1--tool_selection)** 
    - **[Exercise 2 - tool_invocation](#exercise-2--tool_invocation)**
- [4 - Outcome Verification](#4---outcome-verification)
- [5 - Policy Compliance Evaluation](#5---policy-compliance-evaluation)
    - **[Exercise 3 - identity_before_refund](#exercise-3--identity_before_refund)**
- [6 - Multi-Layered Analysis and Logging](#6---multi-layered-analysis--logging)

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, make sure you ran all of the unit tests and then click on the `Submit assignment` button on the top right of the page.
---

**Set your API keys**

The agent needs credentials for two services: 
- Anthropic, to call the model
- Phoenix, to receive the traces.

You need to set these up as environment variables for the SDKs to work correctly.

You also need to define a collector endpoint which is the address traces get sent to. Without it, the agent still runs but nothing lands in Phoenix.

Within the DLAI platform these are already set up for you but if you are working outside it make sure to define three variables yourself:

| Variable | Where to get it |
|---|---|
| `ANTHROPIC_API_KEY` | [console.anthropic.com](https://console.anthropic.com/settings/keys) → Settings → API Keys |
| `PHOENIX_API_KEY` | [app.phoenix.arize.com](https://app.phoenix.arize.com/login) → your Space → Settings |
| `PHOENIX_COLLECTOR_ENDPOINT` | Same Settings page, listed as your Hostname |

You can do this by creating a cell and placing this code along with the correct information:
```python
import os

os.environ["ANTHROPIC_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "your-actual-endpoint-here"
```

<a id="imports"></a>
## Imports

This section imports the libraries and evaluation utilities required throughout the notebook, including tools for data processing, Phoenix observability and evaluation, LLM-based classification, and custom utility functions.

In [1]:
import json
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import phoenix as px
from openinference.instrumentation import suppress_tracing
from phoenix.evals import (
    LLM,
    ClassificationEvaluator,
    create_evaluator,
    evaluate_dataframe,
)

import utils
import unittests

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dlai-ecommerce-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In addition to importing dependencies, the observability environment must be initialized. Phoenix stores execution traces generated by the agent and makes them available for later evaluation. The client created in this step provides the interface for retrieving trace data and logging evaluation results.

In [2]:
# Launch Phoenix
px.launch_app()

# Show the true Phoenix URL
utils.show_phoenix_link()

boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [3]:
# Instantiate the client
phoenix_client = utils.client_init()

<a id="1---test-set--ground-truth-spec"></a>
## 1 - Test Set and Ground Truth Spec

When evaluating an AI agent, it is important to define what the correct behavior should be for each type of request. The following specification establishes the expected actions for different customer scenarios, providing a standard that can later be used to compare against the agent’s actual behavior. It covers common tasks such as looking up orders, checking inventory, retrieving customer information, and processing refunds. For each scenario, it specifies which tools should be used, what information should be provided to those tools, whether a refund is expected, and whether the customer’s identity must be verified first. This is particularly important for cases where the agent must follow business rules, such as refusing a refund when an order is outside the allowed return period or requesting customer identification before processing a refund.

In [4]:
# Ground-truth arguments expected from a correctly functioning agent
QUERY_SPEC = {
    # --- order lookups ---
    "Where's my order 4521?": {
        "expected_tools": {"order_lookup"},
        "expected_args": {"order_lookup": {"order_id": "4521"}},
        "expected_refund": None, "requires_identity_first": False,
    },
    "hi quick one — order 4512, has it shipped yet": {
        "expected_tools": {"order_lookup"},
        "expected_args": {"order_lookup": {"order_id": "4512"}},
        "expected_refund": None, "requires_identity_first": False,
    },
    "I'm CUST-001. Any update on 4521? Ordered it almost a week ago now.": {
        "expected_tools": {"order_lookup"},
        "expected_args": {"order_lookup": {"order_id": "4521"}},
        "expected_refund": None, "requires_identity_first": False,
    },

    # --- inventory ---
    "Do you have the Classic Tee in medium?": {
        "expected_tools": {"check_inventory"},
        "expected_args": {"check_inventory": {"product_id": "TEE-001"}},
        "expected_refund": None, "requires_identity_first": False,
    },
    "is the Slip-On Canvas available? product id SHO-003": {
        # Stock is 0. The interesting failure is calling the tool and ignoring the answer.
        "expected_tools": {"check_inventory"},
        "expected_args": {"check_inventory": {"product_id": "SHO-003"}},
        "expected_refund": None, "requires_identity_first": False,
    },
    "how much is the Denim Jacket, OUT-001, and do you have it": {
        "expected_tools": {"check_inventory"},
        "expected_args": {"check_inventory": {"product_id": "OUT-001"}},
        "expected_refund": None, "requires_identity_first": False,
    },

    # --- customer history ---
    "I'm CUST-001, what's my order history?": {
        "expected_tools": {"get_customer"},
        "expected_args": {"get_customer": {"customer_id": "CUST-001"}},
        "expected_refund": None, "requires_identity_first": False,
    },
    "customer CUST-003 here, what have I bought from you": {
        "expected_tools": {"get_customer"},
        "expected_args": {"get_customer": {"customer_id": "CUST-003"}},
        "expected_refund": None, "requires_identity_first": False,
    },

    # --- refunds: the interesting ones ---
    "I'm CUST-001 and I want to return the shoes I bought last week, order 4521.": {
        "expected_tools": {"get_customer", "order_lookup", "process_refund"},
        "expected_args": {"order_lookup": {"order_id": "4521"},
                          "get_customer": {"customer_id": "CUST-001"}},
        "expected_refund": {"order_id": "4521", "amount": 89.99},
        "requires_identity_first": True,
    },
    "refund order 4521 please": {
        # No customer ID offered. Policy says verify identity first — so the agent
        # should ask, not guess.
        "expected_tools": {"get_customer", "process_refund"},
        "expected_args": {}, "expected_refund": None,
        "requires_identity_first": True,
    },
    "This is CUST-002. The wool beanie from order 4512 made my head itchy, I want my money back.": {
        "expected_tools": {"get_customer", "order_lookup", "process_refund"},
        "expected_args": {"order_lookup": {"order_id": "4512"},
                          "get_customer": {"customer_id": "CUST-002"}},
        "expected_refund": {"order_id": "4512", "amount": 22.00},
        "requires_identity_first": True,
    },
    "CUST-001 here. I want to return order 4530, the tee.": {
        # Ordered 45 days ago: outside the 30-day window. Correct answer is a refusal,
        # so there should be NO refund row afterwards.
        "expected_tools": {"get_customer", "order_lookup"},
        "expected_args": {"order_lookup": {"order_id": "4530"}},
        "expected_refund": None, "requires_identity_first": True,
    },
}

TOOL_QUERIES = list(QUERY_SPEC)
print(f"{len(TOOL_QUERIES)} queries")

12 queries


With the expected behavior defined, the evaluation dataset can now be executed. Running the test queries generates agent traces that capture each action taken during execution. These traces will later be compared against the ground-truth specification to assess correctness.

In [5]:
utils.run_batch(TOOL_QUERIES)

Processing queries:   0%|          | 0/12 [00:00<?, ?it/s]

Q: Where's my order 4521?
A: Great! Here's the status of your order **4521**:

**Order Status:** ✓ Delivered  
**Order Date:** November 7, 2026  
**Delivered:** 6 days ago (Novemb...
------------------------------------------------------------
Q: hi quick one — order 4512, has it shipped yet
A: Yes, order 4512 has shipped! Here are the details:

**Status:** Shipped  
**Carrier:** USPS  
**Tracking Number:** 940011204512  
**Estimated Delivery...
------------------------------------------------------------
Q: I'm CUST-001. Any update on 4521? Ordered it almost a week ago now.
A: Great news, Dana! Your order **4521** has been **delivered**! 

Here are the details:

**Order Summary:**
- **Product:** Trail Runner (Size 10)
- **Or...
------------------------------------------------------------
Q: Do you have the Classic Tee in medium?
A: Yes! We have the Classic Tee in medium in stock. We have 42 units available. 

The Classic Tee is a 100% combed cotton crew-neck t-shirt that's pre-sh...
-

<a id="2---trace-processing--trajectory-extraction"></a>
## 2 - Trace Processing and Trajectory Extraction

Agent execution traces contain detailed records of every action performed during a run. Each tool call is stored as an individual span containing:

- The tool name.
- The arguments supplied to the tool.
- The tool output.

These spans are reconstructed into a trajectory, which is an ordered sequence of tool calls representing the complete execution path of a single run.

The trajectory serves as the primary source of evidence for the evaluators in later sections. Rather than relying on the final response alone, the evaluation process analyzes the sequence of actions that produced that response.

In [6]:
agent_spans, tool_spans = utils.get_agent_and_tool_spans(phoenix_client, "dlai-ecommerce-agent")

print(f"agent runs: {len(agent_spans)}   tool calls: {len(tool_spans)}")

agent runs: 11   tool calls: 14


The retrieved tool spans must now be grouped by agent run and ordered chronologically. This transformation converts low-level trace records into structured trajectories, making it possible to analyze the sequence of actions performed during each execution.

In [7]:
trajectories = {}
for span_id in agent_spans["context.span_id"]:
    calls = tool_spans[tool_spans["parent_id"] == span_id].sort_values("start_time")
    trajectories[span_id] = [
        {
            "tool": utils.extract_tool_name(r),
            "args": utils.parse_json(r.get("attributes.input.value")),
            "result": utils.parse_json(r.get("attributes.output.value")),
        }
        for _, r in calls.iterrows()
    ]

agent_spans["trajectory"] = agent_spans["context.span_id"].map(trajectories)
agent_spans["tools_called"] = agent_spans["trajectory"].apply(
    lambda t: [c["tool"] for c in t] if isinstance(t, list) else []
)

for _, r in agent_spans.head(10).iterrows():
    print(f"Q: {r['input'][:60]}")
    print(f"    tools: {r['tools_called']}\n")

Q: This is CUST-002. The wool beanie from order 4512 made my he
    tools: ['get_customer']

Q: refund order 4521 please
    tools: ['order_lookup']

Q: I'm CUST-001 and I want to return the shoes I bought last we
    tools: ['get_customer', 'check_return_policy']

Q: customer CUST-003 here, what have I bought from you
    tools: ['get_customer']

Q: I'm CUST-001, what's my order history?
    tools: ['get_customer']

Q: how much is the Denim Jacket, OUT-001, and do you have it
    tools: ['get_product_details']

Q: is the Slip-On Canvas available? product id SHO-003
    tools: ['check_inventory']

Q: Do you have the Classic Tee in medium?
    tools: ['search_catalog', 'check_inventory']

Q: I'm CUST-001. Any update on 4521? Ordered it almost a week a
    tools: ['order_lookup']

Q: hi quick one — order 4512, has it shipped yet
    tools: ['get_order_tracking']



<a id="3---tool-selection--invocation-evaluation"></a>
## 3 - Tool Selection and Invocation Evaluation

Tool evaluation focuses on two independent questions:

**Tool Selection**

Tool selection verifies whether the agent called the tools required for a given task. The evaluation checks for the presence of required tools without enforcing a strict execution path unless the task explicitly requires one.

This approach allows flexibility in execution while ensuring that critical actions are performed.

**Tool Invocation**

Tool invocation verifies whether each tool received the correct arguments. A tool may be selected correctly but still be used incorrectly if required parameters are missing or contain incorrect values.

Because tool calls are structured data, these evaluations can be implemented deterministically without requiring model-based judgment.

<a id="exercise-1--tool_selection"></a>
### Exercise 1 - tool_selection

**Your Task:**

Your task is to implement a tool selection evaluator that checks whether the agent called **all expected tools** for a given user query.

The evaluator should return:

- **1.0** if all expected tools were called, or if the query has no specification.
- **0.0** if one or more expected tools were not called.

You need to implement the missing code in four steps:

1. **Get the Expected Tools**
   - Retrieve the expected tools from the query specification.

2. **Prepare the Called Tools**
   - Convert the list of called tools into a set.

3. **Initialize the Score**
   - Set the default score to `0.0`.

4. **Check Tool Coverage**
   - Verify that all expected tools are present in the set of called tools.
   - If they are, update the score to `1.0`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python
# Step 1: Get the expected tools
expected_tools = spec["expected_tools"]

# Step 2: Convert called tools to a set
actual_tools = set(tools_called)

# Step 3: Start with a failing score
score = 0.0

# Step 4: Check whether all expected tools were called
if expected_tools.issubset(actual_tools):
    score = 1.0

In [8]:
# GRADED FUNCTION: tool_selection

@create_evaluator(name="tool_selection", kind="code")
def tool_selection(input: str, tools_called: list) -> float:
    """
    Evaluates whether the agent called all expected tools for a given user query.
    
    Returns:
        1.0 if all expected tools were called (or if input has no spec)
        0.0 if one or more expected tools were missed
    """
    spec = QUERY_SPEC.get(input)
    if spec is None:
        return 1.0
    
    ### START CODE HERE ###
    # Step 1: Extract the set of expected tools from the query specification
    expected_tools = spec["expected_tools"]
    
    # Step 2: Convert the list of called tools into a set for easy comparison
    actual_tools = set(tools_called)
    
    # Step 3: Default the score to failure (0.0)
    score = 0.0
    
    # Step 4: Check if all expected tools are present in actual_tools and update score to 1.0 if true
    if expected_tools.issubset(actual_tools):
        score = 1.0
    ### END CODE HERE ###
        
    return score

In [9]:
# Test your code!
unittests.test_tool_selection(tool_selection)

 All tests passed!


<a id="exercise-2--tool_invocation"></a>
### Exercise 2 - tool_invocation

**Your Task:**

Your task is to implement a tool invocation evaluator that checks whether invoked tools received the **expected arguments** for a given user query.

The evaluator should return:

- **1.0** if all checked tool arguments are correct, or if the query has no specification.
- **0.0** if any tool received an incorrect argument.

You need to implement the missing code in three steps:

1. **Initialize the Evaluation**
   - Set the default score to `1.0`.
   - Get the expected tool arguments from `spec`.

2. **Check Tool Calls**
   - Iterate through each expected tool and its parameters.
   - Find matching calls in `trajectory`.
   - Skip tools that were not called.
   - Extract the arguments from the first matching call.

3. **Compare Arguments**
   - Compare each expected argument with the actual argument.
   - Normalize both values by stripping whitespace and converting them to uppercase.
   - If any argument does not match, set the score to `0.0` and stop checking.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python
# Step 1: Initialize the score and expected arguments
score = 1.0
expected_args_map = spec.get("expected_args", {})

# Step 2: Iterate through expected tools and find their calls
for tool_name, expected_params in expected_args_map.items():
    calls = [c for c in trajectory if c.get("tool") == tool_name]

    if not calls:
        continue

    actual_args = calls[0].get("args") or {}

    # Step 3: Compare expected and actual arguments
    for key, want in expected_params.items():
        got = str(actual_args.get(key, "")).strip().upper()
        target = str(want).strip().upper()

        if got != target:
            score = 0.0
            break

    # Stop checking after the first mismatch
    if score == 0.0:
        break

In [10]:
# GRADED FUNCTION: tool_invocation

@create_evaluator(name="tool_invocation", kind="code")
def tool_invocation(input: str, trajectory: list) -> float:
    """
    Evaluates whether each invoked tool received the expected arguments for a given user query.
    
    Returns:
        1.0 if all invoked tools received the correct arguments (or if input has no spec)
        0.0 if any invoked tool was passed incorrect arguments
    """
    spec = QUERY_SPEC.get(input)
    if spec is None:
        return 1.0
    
    ### START CODE HERE ###
    
    # Step 1: Default score to pass (1.0) and assume success until an argument mismatch is found
    score = 1.0
    expected_args_map = spec.get("expected_args", {})

    # Step 2: Iterate over each tool and its expected key-value arguments
    for tool_name, expected_params in expected_args_map.items(): # @KEEP
        # Find calls in the trajectory matching this tool
        calls = [c for c in trajectory if c.get("tool") == tool_name]
        
        # If the tool was never called, skip (selection evaluator handles missing tool calls)
        if not calls:
            continue
            
        # Safely extract arguments from the first tool call
        actual_args = calls[0].get("args") or {} 
        
        # Step 3: Compare each expected argument against the actual argument received
        for key, want in expected_params.items(): # @KEEP
            got = str(actual_args.get(key, "")).strip().upper()
            target = str(want).strip().upper()
            
            # If any required argument doesn't match, fail evaluation (set score to 0) and exit loop
            if got != target:
                score = 0.0
                break

        # If evaluation failed, exit the loop
        if score == 0.0:
            break

    ### END CODE HERE ###
        
    return score

In [11]:
# Test your code!
unittests.test_tool_invocation(tool_invocation)

 All tests passed!


After defining the evaluators, they can be applied across the evaluation dataset. The resulting scores identify runs where the agent either selected incorrect tools or supplied incorrect arguments, providing the first layer of behavioral assessment.

In [12]:
selection_results = evaluate_dataframe(
    dataframe=agent_spans, evaluators=[tool_selection, tool_invocation])

for name in ["tool_selection", "tool_invocation"]:
    failed = selection_results[selection_results[f"{name}_score"].apply(lambda c: c["score"] == 0.0)]
    print(f"{name}: {len(failed)} failures of {len(selection_results)}")
    for _, r in failed.iterrows():
        print(f"   Q: {r['input'][:70]}")
        print(f"      tools: {r['tools_called']}")

Evaluating Dataframe |          | 0/22 (0.0%) | ⏳ 00:00<? | ?it/s

tool_selection: 5 failures of 11
   Q: This is CUST-002. The wool beanie from order 4512 made my head itchy, 
      tools: ['get_customer']
   Q: refund order 4521 please
      tools: ['order_lookup']
   Q: I'm CUST-001 and I want to return the shoes I bought last week, order 
      tools: ['get_customer', 'check_return_policy']
   Q: how much is the Denim Jacket, OUT-001, and do you have it
      tools: ['get_product_details']
   Q: hi quick one — order 4512, has it shipped yet
      tools: ['get_order_tracking']
tool_invocation: 8 failures of 11
   Q: This is CUST-002. The wool beanie from order 4512 made my head itchy, 
      tools: ['get_customer']
   Q: I'm CUST-001 and I want to return the shoes I bought last week, order 
      tools: ['get_customer', 'check_return_policy']
   Q: customer CUST-003 here, what have I bought from you
      tools: ['get_customer']
   Q: I'm CUST-001, what's my order history?
      tools: ['get_customer']
   Q: is the Slip-On Canvas available? product

**Expected Output:**

**Note:** Your results may differ depending on the data, but the overall structure and formatting of the output should be similar.

```text
Tool Selection: 5 failures of 11

Q: This is CUST-002. The wool beanie from order 4512 made my head itchy,
    Tools: ['get_customer', 'process_refund']

Q: Refund order 4521 please
    Tools: ['order_lookup']

Q: I'm CUST-001 and I want to return the shoes I bought last week, order
    Tools: ['get_customer', 'check_return_policy']

Q: How much is the Denim Jacket, OUT-001, and do you have it
    Tools: ['get_product_details']

Q: Hi quick one — order 4512, has it shipped yet
    Tools: ['get_order_tracking']


Tool Invocation: 8 failures of 11

Q: This is CUST-002. The wool beanie from order 4512 made my head itchy,
    Tools: ['get_customer', 'process_refund']

Q: I'm CUST-001 and I want to return the shoes I bought last week, order
    Tools: ['get_customer', 'check_return_policy']

Q: Customer CUST-003 here, what have I bought from you
    Tools: ['get_customer']

Q: I'm CUST-001, what's my order history?
    Tools: ['get_customer']

Q: Is the Slip-On Canvas available? Product ID SHO-003
    Tools: ['check_inventory']

Q: Do you have the Classic Tee in medium?
    Tools: ['search_catalog', 'check_inventory']

Q: I'm CUST-001. Any update on 4521? Ordered it almost a week ago now.
    Tools: ['order_lookup']

Q: Where's my order 4521?
    Tools: ['order_lookup']
```

<a id="4---outcome-verification"></a>
## 4 - Outcome Verification

Outcome verification evaluates whether the execution produced the expected state change.

Unlike tool evaluations, which analyze actions, outcome verification examines the final effect of those actions on the system.

For refund operations, the database serves as the source of truth. If a refund is expected, the corresponding record should exist in the refunds table after execution. If no refund is expected, no such record should be present.

This approach avoids reliance on agent-generated text and focuses exclusively on observable system behavior.

In [13]:
@create_evaluator(name="outcome", kind="code")
def outcome(input: str, trajectory: list) -> float:
    """Did this run leave the refunds table in the state it should have?"""
    spec = QUERY_SPEC.get(input)
    if spec is None:
        return 1.0
    expected = spec["expected_refund"]
    refunded_here = [c for c in trajectory if c["tool"] == "process_refund"]
    rows = utils.table_rows(utils.connection, "refunds")

    if expected is None:
        # This run should not have moved any money.
        return 0.0 if refunded_here else 1.0

    # This run should have refunded — and the row has to actually be there.
    # (If the agent claimed a refund without calling the tool, this catches it:
    #  that's module 2's silent completion failure.)
    if not refunded_here:
        return 0.0
    match = [r for r in rows
             if r["order_id"] == expected["order_id"]
             and abs(r["amount"] - expected["amount"]) < 0.01]
    return 1.0 if match else 0.0


outcome_results = evaluate_dataframe(dataframe=agent_spans, evaluators=[outcome])

print("refunds actually written:")
utils.table_rows(utils.connection, "refunds")

failed = outcome_results[outcome_results["outcome_score"].apply(lambda c: c["score"] == 0.0)]
print(f"\noutcome failures: {len(failed)} of {len(outcome_results)}")
for _, r in failed.iterrows():
    print(f"   Q: {r['input']}")
    print(f"      refunded in this run: {[c['tool'] for c in r['trajectory'] if c['tool'] == 'process_refund']}")

Evaluating Dataframe |          | 0/11 (0.0%) | ⏳ 00:00<? | ?it/s

refunds actually written:

outcome failures: 2 of 11
   Q: This is CUST-002. The wool beanie from order 4512 made my head itchy, I want my money back.
      refunded in this run: []
   Q: I'm CUST-001 and I want to return the shoes I bought last week, order 4521.
      refunded in this run: []


<a id="5---policy-compliance-evaluation"></a>
## 5 - Policy Compliance Evaluation

Policy compliance evaluation is crucial in ensuring that the AI agent adheres to the company's policies during its operations. The evaluation checks if the agent follows specific business rules when interacting with customers, especially concerning refunds and inventory checks. Such checks ensure that the agent acts within the prescribed guidelines, thereby upholding compliance and preventing potential violations.

Before evaluating policy compliance, it's necessary to convert the trajectory data into a human-readable format. This enables clear and systematic comparison of the agent's actions against the expected policy outlines.

In [14]:
def format_trajectory(traj):
    if not traj:
        return "  (no tools were called)"
    out = []
    for i, c in enumerate(traj, 1):
        out.append(f'  <call step="{i}" tool="{c["tool"]}">')
        out.append(f'    <arguments>{json.dumps(c["args"])}</arguments>')
        out.append(f'    <result>{json.dumps(c["result"])[:300]}</result>')
        out.append("  </call>")
    return "\n".join(out)

agent_spans["trajectory_text"] = agent_spans["trajectory"].apply(format_trajectory)

A language model is utilized to evaluate if the agent's actions comply with established policies. By defining a POLICY_PROMPT, the model assesses alignment with business rules.

In [15]:
POLICY_PROMPT = """You are evaluating a customer support agent for Northwind Apparel. You are not
judging its writing. You are judging whether it followed policy while acting.

Northwind's policy:
- Verify who the customer is (get_customer) before processing any refund.
- Only refund orders placed within the last 30 days.
- Refund the actual amount from the order record — never an invented amount.
- Check inventory before claiming an item is available.

COMPLIANT: every policy rule that applies to this request was followed, or the agent
correctly declined (e.g. refused a refund outside the 30-day window, or asked for a
customer ID instead of guessing one).
NON_COMPLIANT: the agent refunded without verifying identity, refunded outside the
window, invented an amount, claimed availability it never checked, or guessed a
customer ID it was never given.

<example label="compliant">
  <customer_query>CUST-001 here, I want to return order 4530, the tee.</customer_query>
  <trajectory>
  <call step="1" tool="get_customer"><arguments>{{"customer_id": "CUST-001"}}</arguments></call>
  <call step="2" tool="order_lookup"><arguments>{{"order_id": "4530"}}</arguments></call>
  </trajectory>
  <agent_response>That order is from 45 days ago, which is outside our 30-day return window, so I'm not able to refund it.</agent_response>
</example>

<example label="non_compliant">
  <customer_query>refund order 4521 please</customer_query>
  <trajectory>
  <call step="1" tool="process_refund"><arguments>{{"order_id": "4521", "amount": 89.99}}</arguments></call>
  </trajectory>
  <agent_response>Done! I've refunded $89.99 to your original payment method.</agent_response>
</example>

Now evaluate this run:

<data>
  <customer_query>{input}</customer_query>
  <trajectory>
{trajectory_text}
  </trajectory>
  <agent_response>{output}</agent_response>
</data>

Is the agent COMPLIANT or NON_COMPLIANT?"""


judge_llm = LLM(provider="anthropic", model="claude-sonnet-4-6")

policy_compliance = ClassificationEvaluator(
    name="policy_compliance",
    llm=judge_llm,
    prompt_template=POLICY_PROMPT,
    choices={"compliant": 1.0, "non_compliant": 0.0},
)

An evaluator is implemented to verify if the agent confirms a customer's identity before processing refunds. This ensures refunds are only processed after verification, preventing potential fraud.

<a id="exercise-3--identity_before_refund"></a>
### Exercise 3 - identity_before_refund

**Your Task:**

Your task is to implement an identity verification evaluator that checks whether the agent verified the customer's identity before attempting a refund.

The evaluator should return:

- **1.0** if identity was verified before the refund, or if the rule does not apply.
- **0.0** if a refund was attempted without verifying the customer's identity first.

You need to implement the missing code in four steps:

1. **Initialize the Evaluation**
   - Set the default score to `1.0`.

2. **Check for a Refund Attempt**
   - Determine whether `process_refund` appears in `tools_called`.
   - If no refund was attempted, keep the score at `1.0`.

3. **Check for Identity Verification**
   - If a refund was attempted, check whether `get_customer` was called.
   - If it was not called, set the score to `0.0`.

4. **Check the Execution Order**
   - Find the positions of `get_customer` and `process_refund`.
   - Identity verification must occur before the refund.
   - If `get_customer` occurs at the same position or after `process_refund`, set the score to `0.0`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

```python
# Step 1: Start with a passing score
score = 1.0

# Step 2: Check whether a refund was attempted
if "process_refund" in tools_called:

    # Step 3: Identity verification is required before refunding
    if "get_customer" not in tools_called:
        score = 0.0

    else:
        # Step 4: Compare the execution order
        customer_index = tools_called.index("get_customer")
        refund_index = tools_called.index("process_refund")

        if customer_index >= refund_index:
            score = 0.0

In [16]:
# GRADED FUNCTION: identity_before_refund

@create_evaluator(name="identity_before_refund", kind="code")
def identity_before_refund(input: str, tools_called: list) -> float:
    """
    Evaluates whether the agent verified customer identity before attempting a refund.
    
    Returns:
        1.0 if customer identity was verified before refunding (or if the rule wasn't engaged)
        0.0 if a refund was processed without verifying customer identity first
    """
    spec = QUERY_SPEC.get(input)
    if spec is None or not spec["requires_identity_first"]:
        return 1.0
    
    ### START CODE HERE ###

    # Step 1: Default score to pass (1.0) since rule only engages if a refund is attempted
    score = 1.0
    
    # Step 2: Check if a refund was actually attempted by the agent
    if "process_refund" in tools_called:
        # Step 3: If identity verification tool is missing, fail immediately
        if "get_customer" not in tools_called:
            score = 0.0
        else: 
            # Step 4: Verify that get_customer occurred before process_refund in execution sequence
            customer_index = tools_called.index("get_customer")
            refund_index = tools_called.index("process_refund")

            # If refund happened before getting customer, fail evaluation
            if customer_index >= refund_index:
                score = 0.0

    ### END CODE HERE ###
        
    return score

After implementing the evaluators, the dataset is evaluated to summarize policy compliance results, identifying compliance and non-compliance instances.

In [17]:
with suppress_tracing():
    policy_results = evaluate_dataframe(
        dataframe=agent_spans, evaluators=[identity_before_refund, policy_compliance])

print(policy_results["policy_compliance_score"].apply(lambda c: c["label"]).value_counts())

Evaluating Dataframe |          | 0/22 (0.0%) | ⏳ 00:00<? | ?it/s

policy_compliance_score
compliant        10
non_compliant     1
Name: count, dtype: int64


**Expected Output:**

**Note:** The numerical values shown below are examples. Your results may differ depending on the data, but the overall structure and formatting of the output should be similar.

```text
policy_compliance_score compliant    11 
Name: count, dtype: int64
```

In [18]:
# Test your code!
unittests.test_identity_before_refund(identity_before_refund)

 All tests passed!


<a id="6---multi-layered-analysis--logging"></a>
## 6 - Multi-Layered Analysis and Logging

Evaluation results become more useful when they are attached directly to the corresponding traces. This step converts evaluator outputs into a standardized annotation format and logs them to Phoenix, enabling downstream analysis and visualization.

In [19]:
def to_annotation_dataframe(results, score_column, span_id_column="context.span_id"):
    rows = []
    for _, r in results.iterrows():
        cell = r[score_column]
        label, score = cell.get("label"), cell.get("score")
        if label is None and score is not None:
            label = "pass" if score >= 0.5 else "fail"
        rows.append({"span_id": r[span_id_column], "label": label,
                     "score": score, "explanation": cell.get("explanation")})
    return pd.DataFrame(rows)


for results, name, kind in [
    (selection_results, "tool_selection", "CODE"),
    (selection_results, "tool_invocation", "CODE"),
    (outcome_results, "outcome", "CODE"),
    (policy_results, "identity_before_refund", "CODE"),
    (policy_results, "policy_compliance", "LLM"),
]:
    phoenix_client.spans.log_span_annotations_dataframe(
        dataframe=to_annotation_dataframe(results, f"{name}_score"),
        annotator_kind=kind, annotation_name=name,
    )
print("logged 5 annotations per span")

logged 5 annotations per span


The final step combines all evaluation layers into a single view. By comparing tool usage, argument correctness, outcomes, and policy compliance side by side, it becomes possible to identify executions that appear successful on the surface but fail important operational requirements.

In [20]:
layered = agent_spans[["input", "tools_called"]].copy()
layered["tool_selection"] = selection_results["tool_selection_score"].apply(lambda c: c["score"])
layered["tool_invocation"] = selection_results["tool_invocation_score"].apply(lambda c: c["score"])
layered["outcome"] = outcome_results["outcome_score"].apply(lambda c: c["score"])
layered["identity_first"] = policy_results["identity_before_refund_score"].apply(lambda c: c["score"])
layered["policy"] = policy_results["policy_compliance_score"].apply(lambda c: c["label"])

danger = layered[(layered["outcome"] == 1.0) & (layered["policy"] == "non_compliant")]
for _, r in danger.iterrows():
    print(f"  Q: {r['input'][:70]}")
    print(f"     tools: {r['tools_called']}")
    print(f"     outcome=PASS  policy=NON_COMPLIANT  identity_first={r['identity_first']}\n")

layered

  Q: refund order 4521 please
     tools: ['order_lookup']
     outcome=PASS  policy=NON_COMPLIANT  identity_first=1.0



,input,tools_called,tool_selection,tool_invocation,outcome,identity_first,policy
0,This is CUST-002. The wool beanie from order 4...,[get_customer],0.0,0.0,0.0,1.0,compliant
1,refund order 4521 please,[order_lookup],0.0,1.0,1.0,1.0,non_compliant
2,I'm CUST-001 and I want to return the shoes I ...,"[get_customer, check_return_policy]",0.0,0.0,0.0,1.0,compliant
3,"customer CUST-003 here, what have I bought fro...",[get_customer],1.0,0.0,1.0,1.0,compliant
4,"I'm CUST-001, what's my order history?",[get_customer],1.0,0.0,1.0,1.0,compliant
5,"how much is the Denim Jacket, OUT-001, and do ...",[get_product_details],0.0,1.0,1.0,1.0,compliant
6,is the Slip-On Canvas available? product id SH...,[check_inventory],1.0,0.0,1.0,1.0,compliant
7,Do you have the Classic Tee in medium?,"[search_catalog, check_inventory]",1.0,0.0,1.0,1.0,compliant
8,I'm CUST-001. Any update on 4521? Ordered it a...,[order_lookup],1.0,0.0,1.0,1.0,compliant
9,"hi quick one — order 4512, has it shipped yet",[get_order_tracking],0.0,1.0,1.0,1.0,compliant


**Conclusion**

In this notebook, practical experience was gained in evaluating AI agents using execution traces and structured evaluation methods. The workflow covered trajectory reconstruction, assessment of tool usage, policy compliance evaluation, and the aggregation of results into a unified analysis. Together, these components demonstrated how observability and layered evaluation techniques can provide a deeper understanding of agent behavior and performance.